# 🚀 Baidu Unlimited-OCR FastAPI Microservice on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arslaan7861/unlimited-ocr/blob/master/colab_unlimited_ocr.ipynb)

This notebook runs the complete **Baidu Unlimited-OCR** FastAPI microservice on Google Colab with GPU acceleration and exposes it via an **ngrok** public tunnel.

### Instructions:
1. **Enable GPU**: Click `Runtime` -> `Change runtime type` -> select `T4 GPU` (or `A100` / `L4`).
2. **Ngrok Token**: Grab your free authtoken from [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) (or add it to Colab Secrets as `NGROK_AUTHTOKEN`).
3. **Run**: Click `Runtime` -> `Run all`.

### Step 1: Verify NVIDIA GPU Acceleration

In [ ]:
!nvidia-smi

### Step 2: Clone Repository

In [ ]:
import os
if not os.path.exists("app"):
    !git clone https://github.com/arslaan7861/unlimited-ocr.git
    %cd unlimited-ocr
else:
    print("Already inside repository directory.")

### Step 3: Install Lightweight Dependencies & ngrok

In [ ]:
# Colab pre-installs PyTorch and CUDA. We only install the web and OCR helper packages.
!pip install -q fastapi "uvicorn[standard]" pydantic-settings python-multipart pymupdf transformers addict einops easydict pyngrok

### Step 4: Download Baidu Unlimited-OCR Model Weights

In [ ]:
import os
os.makedirs("models/unlimited-ocr", exist_ok=True)
!huggingface-cli download baidu/Unlimited-OCR --local-dir models/unlimited-ocr

### Step 5: Configure Production Environment for CUDA GPU

In [ ]:
env_content = """# Colab GPU Configuration
HOST=0.0.0.0
PORT=8000
ENVIRONMENT=production
LOG_LEVEL=INFO
MODEL_PATH=models/unlimited-ocr
DEVICE=cuda
TORCH_DTYPE=bfloat16
MAX_UPLOAD_SIZE_MB=25
MAX_PDF_PAGES=50
MAX_CONCURRENT_INFERENCES=4
PDF_DPI=300
"""

with open(".env", "w", encoding="utf-8") as f:
    f.write(env_content)

print(".env successfully configured for CUDA inference.")

### Step 6: Connect ngrok & Start the FastAPI Server

If you added `NGROK_AUTHTOKEN` to your **Colab Secrets** (key icon in the left sidebar), it will be used automatically. Otherwise, you'll be prompted to enter it.

In [ ]:
import getpass
from pyngrok import ngrok

# 1. Retrieve or prompt for ngrok token
try:
    from google.colab import userdata
    ngrok_token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    ngrok_token = None

if not ngrok_token:
    ngrok_token = getpass.getpass("Enter your ngrok auth token (from https://dashboard.ngrok.com/get-started/your-authtoken): ")

ngrok.set_auth_token(ngrok_token)
ngrok.kill()  # Close any previously open tunnels

# 2. Establish public HTTPS tunnel to port 8000
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("=" * 65)
print(f"🚀 PUBLIC API URL:     {public_url}")
print(f"📖 INTERACTIVE DOCS:   {public_url}/docs")
print(f"🩺 HEALTH PROBE:       {public_url}/health")
print("=" * 65)

# 3. Run FastAPI microservice
!uvicorn app.main:app --host 0.0.0.0 --port 8000